# Import Library

In [ ]:
import re
import random
import warnings
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import joblib

import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

warnings.filterwarnings("ignore")

# Set Random Seed

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Download NLTK Resource

In [3]:
try:
    nltk.download("punkt_tab")
except Exception:
    pass

try:
    nltk.download("punkt")
except Exception:
    pass

try:
    nltk.download("stopwords")
except Exception:
    pass

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Path dan Konfigurasi

In [4]:
DATA_PATH = Path("dataset/dataset.csv")   # ubah sesuai lokasi dataset Anda
TEXT_COL = "tweet"                        # ubah kalau nama kolom teks berbeda
LABEL_COL = "category"                    # ubah kalau nama kolom label berbeda

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Load Dataset

In [5]:
df = pd.read_csv(DATA_PATH)

# pastikan kolom ada
assert TEXT_COL in df.columns, f"Kolom teks '{TEXT_COL}' tidak ditemukan."
assert LABEL_COL in df.columns, f"Kolom label '{LABEL_COL}' tidak ditemukan."

# buang data kosong
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# ubah ke string
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(str)

print("Jumlah data:", len(df))
print(df.head())

Jumlah data: 48704
                                               tweet       category
0  Nest Protect mendapat penarikan kembali dan pe...     Technology
1                 PS4 vs. Xbox One: 6 Bulan Kemudian     Technology
2                                   Pertemuan Gereja  Entertainment
3  Facebook: Kami "tidak punya rencana" untuk men...     Technology
4  Kudeta di Thailand memicu pembatalan pemesanan...  Entertainment


# Encoding Label

In [6]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df[LABEL_COL])

class_names = list(le.classes_)
print("Class names:", class_names)

Class names: ['Economy', 'Entertainment', 'Technology']


In [10]:
joblib.dump(le, "artifacts/label_encoder.pkl")

['artifacts/label_encoder.pkl']

# Preprocessing

In [7]:
stopword_factory = StopWordRemoverFactory()
indonesian_stopwords = set(stopword_factory.get_stop_words())

# tambahan stopword umum Twitter / dataset informal
extra_stopwords = {
    "rt", "via", "yg", "dgn", "nya", "nih", "deh", "dong", "sih",
    "lah", "kok", "ya", "ga", "gak", "nggak", "aja", "dll", "dsb"
}
indonesian_stopwords |= extra_stopwords

stemmer_factory = StemmerFactory()
indo_stemmer = stemmer_factory.create_stemmer()

def clean_basic_text(text: str) -> str:
    """
    Pembersihan awal:
    - lowercase
    - hapus URL
    - hapus mention
    - hapus karakter non-alfanumerik tertentu
    - rapikan spasi
    """
    text = str(text).lower()

    # hapus url
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # hapus mention
    text = re.sub(r"@\w+", " ", text)

    # hapus hashtag symbol tapi sisakan katanya
    text = text.replace("#", " ")

    # hapus karakter selain huruf/angka/spasi
    # (emoji akan ikut hilang; kalau mau dipertahankan, ubah regex ini)
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()

    return text

def preprocess_classical(text: str) -> str:
    """
    Pipeline sesuai PDF acuan:
    1) Lowercase
    2) Remove URL
    3) Remove punctuation/special char
    4) Tokenization
    5) Stopword removal
    6) Stemming Bahasa Indonesia
    """
    text = clean_basic_text(text)
    tokens = word_tokenize(text)

    tokens = [t for t in tokens if t not in indonesian_stopwords and len(t) > 1]
    tokens = [indo_stemmer.stem(t) for t in tokens]
    tokens = [t for t in tokens if t.strip()]

    return " ".join(tokens)

# contoh uji
sample = "Saya suka teknologi AI! Cek https://openai.com #AI #teknologi"
print("Before :", sample)
print("After  :", preprocess_classical(sample))

# terapkan preprocessing
df["text_clean"] = df[TEXT_COL].apply(preprocess_classical)

print(df[[TEXT_COL, "text_clean", LABEL_COL]].head())

Before : Saya suka teknologi AI! Cek https://openai.com #AI #teknologi
After  : suka teknologi ai cek ai teknologi
                                               tweet  \
0  Nest Protect mendapat penarikan kembali dan pe...   
1                 PS4 vs. Xbox One: 6 Bulan Kemudian   
2                                   Pertemuan Gereja   
3  Facebook: Kami "tidak punya rencana" untuk men...   
4  Kudeta di Thailand memicu pembatalan pemesanan...   

                                          text_clean       category  
0       nest protect dapat tari baru perangkat lunak     Technology  
1                     ps4 vs xbox one bulan kemudian     Technology  
2                                        temu gereja  Entertainment  
3    facebook punya rencana desain ulang oculus rift     Technology  
4  kudeta thailand picu batal mesan baru turis ne...  Entertainment  


In [12]:
def preprocess_for_bert(text):
    text = str(text).lower()

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    text = re.sub(r"@\w+", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

df["text_bert"] = df[TEXT_COL].apply(preprocess_for_bert)

# Splitting Data

In [15]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df["label_encoded"]
)

print("Train size:", len(train_df))
print("Test size :", len(test_df))

Train size: 38963
Test size : 9741


## Dataset untuk TF-IDF dan FastText

In [17]:
X_train = train_df["text_clean"]
X_test = test_df["text_clean"]

y_train = train_df["label_encoded"]
y_test = test_df["label_encoded"]

# Dataset untuk BERT

In [18]:
train_df_bert = train_df[
    ["text_bert", "label_encoded"]
].rename(
    columns={
        "text_bert":"text",
        "label_encoded":"label"
    }
)

test_df_bert = test_df[
    ["text_bert", "label_encoded"]
].rename(
    columns={
        "text_bert":"text",
        "label_encoded":"label"
    }
)

In [19]:
os.makedirs("dataset/preprocessed", exist_ok=True)

# Classical
X_train.to_csv(
    "dataset/preprocessed/X_train.csv",
    index=False
)

X_test.to_csv(
    "dataset/preprocessed/X_test.csv",
    index=False
)

y_train.to_csv(
    "dataset/preprocessed/y_train.csv",
    index=False
)

y_test.to_csv(
    "dataset/preprocessed/y_test.csv",
    index=False
)

# BERT
train_df_bert.to_csv(
    "dataset/preprocessed/train_bert.csv",
    index=False
)

test_df_bert.to_csv(
    "dataset/preprocessed/test_bert.csv",
    index=False
)